# 모듈 1: 인공신경망의 태동과 퍼셉트론 (Perceptron) 실습

이 노트북은 전공자 및 비전공자가 파이토치(PyTorch)를 활용하여
퍼셉트론의 핵심인 **선형 결합**과 **계단 함수**를 직관적으로 이해하고 증명하기 위해 제작되었습니다.

In [ ]:
import torch

# CPU 환경 기준 세팅
torch.manual_seed(42)
print(f"PyTorch Version: {torch.__version__}")

## 1. 계단 함수 (Step Function) 정의

계단 함수는 특정 임계치(여기서는 0)를 넘을 경우에만 신호를 전달(1)하도록 하는 스위치입니다.

In [ ]:
def step_function(x):
    # x가 0 이상이면 1.0, 0 미만이면 0.0을 반환
    return (x >= 0).float()

## 2. 단층 퍼셉트론 논리 증명: AND 게이트

AND 게이트는 두 입력이 모두 1일때만 1을 출력합니다.
우리가 가중치($w_1, w_2$)를 각각 0.5로, 편향($b$)을 -0.7로 임의 설정해보겠습니다.
$$ y = 0.5x_1 + 0.5x_2 - 0.7 $$
이 결과가 계단 함수를 통과하면 완벽하게 AND 논리가 성립됩니다.

In [ ]:
# 입력 데이터 (0,0), (0,1), (1,0), (1,1)
x = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]])

# 가중치와 편향
w_and = torch.tensor([[0.5], [0.5]])
b_and = -0.7

# 선형 결합 식 적용 (행렬곱 + 편향)
y_sum_and = torch.matmul(x, w_and) + b_and

# 계단 함수 통과
output_and = step_function(y_sum_and)

print("--- AND Gate 테스트 결과 ---")
for x_val, out in zip(x, output_and):
    print(f"입력 {x_val.tolist()} -> 출력 {int(out.item())}")

## 3. XOR 게이트의 실패 (선형 분리 불가능성 증명)

XOR 논리는 입력이 서로 다를 때만 1을 출력합니다.
어떤 가중치와 편향을 넣더라도 직선 하나(단층 퍼셉트론)로는 이 논리를 완벽히 가를 수 없음을 확인합니다.

In [ ]:
# 임의의 가중치로 XOR을 시도해봅니다.
w_xor = torch.tensor([[1.0], [1.0]])
b_xor = -0.5

y_sum_xor = torch.matmul(x, w_xor) + b_xor
output_xor = step_function(y_sum_xor)

print("--- XOR Gate 시도 결과 (실패) ---")
for x_val, out in zip(x, output_xor):
    correct_target = int(x_val[0] != x_val[1])
    print(f"입력 {x_val.tolist()} -> 출력 {int(out.item())} (실제 정답: {correct_target})")

print("\n-> 결과 분석: 퍼셉트론 하나(단일 선형 결합)로는 XOR과 같은 비선형 문제를 분리하지 못합니다. 이것이 다음 모듈에서 '은닉층(Hidden Layer)'을 추가하는 이유입니다.")

## 4. 공식 적용 예제: 대출 승인 퍼셉트론

공식 $y = step(w_1x_1 + w_2x_2 + b)$ 를 실제 의사결정 예제로 적용합니다.

- 입력: `x1=소득(정규화)`, `x2=기존 연체 위험(정규화)`
- 해석: 선형결합 점수가 0 이상이면 승인(1), 아니면 거절(0)


In [ ]:
# 공식 적용: y = step(w1*x1 + w2*x2 + b)
def step(x):
    return 1 if x >= 0 else 0

w1, w2, b = 0.8, -1.2, -0.1
samples = [
    (0.9, 0.1),  # 고소득, 저위험
    (0.4, 0.7),  # 중소득, 고위험
    (0.7, 0.5),
]

for i, (x1, x2) in enumerate(samples, 1):
    z = w1 * x1 + w2 * x2 + b
    y = step(z)
    print(f'샘플{i}: z={z:.3f}, 예측={y}')
